# 07c — Entrenamiento split por bloques + normalización propia (v2_tuned)

Segunda corrida sobre el **mismo split** de `06_split_bloques.ipynb` (`v2_bloques`), cambiando
únicamente la normalización de entrada: en lugar de las constantes ImageNet se usan las medidas
sobre train en `03b_normalizacion_chips.ipynb` (`data/08_reporting/normalization_constants.json`).

`ColorJitter` se mantiene (03b no encontró degradación medible de la señal fotométrica).
No se modifica `src/data/dataset.py` ni los notebooks `07`/`08` originales.

Salidas bajo `v2_bloques_tuned/`:

```text
data/06_models/v2_bloques_tuned/{arquitectura}_best.pt
data/08_reporting/v2_bloques_tuned/...
reports/figures/{arquitectura}_v2_bloques_tuned_training_curves.png
```

Manifiestos reutilizados de `data/05_model_input/v2_bloques/` (no reejecutar el 06).


## 1. Configuración

In [ ]:
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch.utils.data import DataLoader

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from src.data.dataset import GarimpoDataset, load_manifest
from src.models.train import train_model
from src.utils.helpers import get_device, set_seed

RUN_NAME = "v2_bloques_tuned"
SPLIT_NAME = "v2_bloques"  # manifiestos del 06; no reentrenar split

DATA_DIR = ROOT / "data"
MANIFEST_DIR = DATA_DIR / "05_model_input" / SPLIT_NAME
MODELS_DIR = DATA_DIR / "06_models" / RUN_NAME
REPORT_DIR = DATA_DIR / "08_reporting" / RUN_NAME
FIGURES_DIR = ROOT / "reports" / "figures"
MODELS_DIR = DATA_DIR / "06_models" / RUN_NAME
REPORT_DIR = DATA_DIR / "08_reporting" / RUN_NAME
FIGURES_DIR = ROOT / "reports" / "figures"

# La carpeta de chips no está en el mismo sitio en todas las máquinas del proyecto
CHIPS_DIR_CANDIDATOS = [
    DATA_DIR / "01_raw" / "dataset_amazonia_garimpo_binario",
    DATA_DIR / "Dataset" / "datasets" / "amazonia_garimpo" / "dataset_amazonia_garimpo_binario",
]
CHIPS_DIR = next((p for p in CHIPS_DIR_CANDIDATOS if p.exists()), None)
if CHIPS_DIR is None:
    rutas = "\n  ".join(str(p) for p in CHIPS_DIR_CANDIDATOS)
    raise FileNotFoundError(f"No se encontró la carpeta de chips. Probadas:\n  {rutas}")

BATCH_SIZE = 32
NUM_WORKERS = 4
MAX_EPOCHS = 30
PATIENCE = 7
LR_CNN = 1e-4
LR_TRANSFORMER = 3e-5
SEED = 42

MODELS = [
    {"name": "efficientnet_b0",              "label": "EfficientNet-B0 (v2 bloques tuned)",  "lr": LR_CNN},
    {"name": "resnet50",                     "label": "ResNet-50 (v2 bloques tuned)",        "lr": LR_CNN},
    {"name": "swin_tiny_patch4_window7_224", "label": "Swin Transformer (v2 bloques tuned)", "lr": LR_TRANSFORMER},
    {"name": "vit_tiny_patch16_224",         "label": "ViT tiny (v2 bloques tuned)",         "lr": LR_TRANSFORMER},
]

MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

set_seed(SEED)
device = get_device()
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else str(device)

print(f"Corrida: {RUN_NAME}")
print(f"Device: {device}")
print(f"GPU: {gpu_name}")
print(f"PyTorch: {torch.__version__}")
print(f"Batch={BATCH_SIZE} | Max epochs={MAX_EPOCHS} | Patience={PATIENCE}")
print(f"LR CNN={LR_CNN} | LR Transformer={LR_TRANSFORMER}")
print(f"Chips       : {CHIPS_DIR}")
print(f"Manifiestos : {MANIFEST_DIR}")
print(f"Checkpoints : {MODELS_DIR}")


## 2. Transforms y DataLoaders

Los manifiestos son los de `v2_bloques` (split ya validado). Los transforms viven **en este
notebook** con `DATASET_MEAN` / `DATASET_STD` medidos sobre train. `ColorJitter` se conserva
igual que en la corrida v2.


In [ ]:
import json
from torchvision import transforms

NORM_JSON = DATA_DIR / "08_reporting" / "normalization_constants.json"
if not NORM_JSON.exists():
    raise FileNotFoundError(
        f"No se encontró {NORM_JSON}. Ejecuta antes 03b_normalizacion_chips.ipynb."
    )
_norm = json.loads(NORM_JSON.read_text(encoding="utf-8"))
DATASET_MEAN = tuple(_norm["DATASET_MEAN"])
DATASET_STD = tuple(_norm["DATASET_STD"])
print(f"Normalización propia (desde {NORM_JSON.name}):")
print(f"  DATASET_MEAN = {DATASET_MEAN}")
print(f"  DATASET_STD  = {DATASET_STD}")


def get_transforms_tuned(split: str) -> transforms.Compose:
    """Mismos augments que get_transforms(), con normalización medida sobre train."""
    normalize = transforms.Normalize(mean=DATASET_MEAN, std=DATASET_STD)
    if split == "train":
        return transforms.Compose([
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomRotation(degrees=90),
            transforms.ColorJitter(
                brightness=0.2,
                contrast=0.2,
                saturation=0.1,
                hue=0.05,
            ),
            transforms.ToTensor(),
            normalize,
        ])
    return transforms.Compose([
        transforms.ToTensor(),
        normalize,
    ])


In [ ]:
def crear_dataloaders(
    manifest_dir: Path,
    chips_dir: Path,
    batch_size: int,
    num_workers: int,
) -> dict[str, DataLoader]:
    """DataLoaders a partir de los manifiestos de una corrida concreta."""
    loaders: dict[str, DataLoader] = {}
    for split in ("train", "val", "test"):
        manifest = load_manifest(manifest_dir / f"manifest_{split}.csv")
        dataset = GarimpoDataset(
            manifest=manifest,
            chips_dir=chips_dir,
            transform=get_transforms_tuned(split),
        )
        loaders[split] = DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=(split == "train"),
            num_workers=num_workers,
            pin_memory=True,
            persistent_workers=(num_workers > 0),
        )
    return loaders


loaders = crear_dataloaders(MANIFEST_DIR, CHIPS_DIR, BATCH_SIZE, NUM_WORKERS)

train_loader = loaders["train"]
val_loader = loaders["val"]
test_loader = loaders["test"]

for split, loader in loaders.items():
    manifest = loader.dataset.manifest
    print(
        f"{split:<5} {len(manifest):>7,} chips | "
        f"{len(loader):>5,} batches | "
        f"com_garimpo {manifest['label_int'].mean() * 100:5.1f} %"
    )


## 3. Helpers — gráficos y exportación

In [ ]:
def history_to_dataframe(summary: dict) -> pd.DataFrame:
    """Combina métricas train/val por epoch en un DataFrame."""
    train_df = pd.DataFrame(summary["history"]["train"]).add_prefix("train_")
    val_df = pd.DataFrame(summary["history"]["val"]).add_prefix("val_")
    if "train_epoch" in train_df.columns:
        train_df = train_df.rename(columns={"train_epoch": "epoch"})
    if "val_epoch" in val_df.columns:
        val_df = val_df.drop(columns=["val_epoch"])
    df = pd.concat([train_df, val_df], axis=1)
    df.insert(0, "model", summary.get("model_label", summary["model_name"]))
    df.insert(1, "timm_name", summary["model_name"])
    return df


def plot_training_curves(summary: dict, save_path: Path) -> None:
    history = summary["history"]
    label = summary.get("model_label", summary["model_name"])
    epochs = [r["epoch"] for r in history["train"]]

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].plot(epochs, [r["loss"] for r in history["train"]], label="train")
    axes[0].plot(epochs, [r["loss"] for r in history["val"]], label="val")
    axes[0].set_title("Loss"); axes[0].legend(); axes[0].set_xlabel("epoch")

    axes[1].plot(epochs, [r["accuracy"] for r in history["train"]], label="train")
    axes[1].plot(epochs, [r["accuracy"] for r in history["val"]], label="val")
    axes[1].set_title("Accuracy"); axes[1].legend(); axes[1].set_xlabel("epoch")

    axes[2].plot(epochs, [r["macro_f1"] for r in history["val"]], label="val macro F1", color="tomato")
    axes[2].plot(epochs, [r["recall_com_garimpo"] for r in history["val"]], label="val recall com_garimpo")
    axes[2].set_title("Val F1 / Recall com_garimpo"); axes[2].legend(); axes[2].set_xlabel("epoch")

    plt.suptitle(f"{label} — curvas de entrenamiento", y=1.02)
    plt.tight_layout()
    save_path.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()


def format_epoch_line(epoch: int, max_ep: int, train_r: dict, val_r: dict, elapsed: float) -> str:
    return (
        f"Epoch {epoch:02d}/{max_ep} | "
        f"train loss {train_r['loss']:.4f} acc {train_r['accuracy']:.3f} | "
        f"val loss {val_r['loss']:.4f} acc {val_r['accuracy']:.3f} "
        f"F1 {val_r['macro_f1']:.3f} "
        f"(com recall {val_r['recall_com_garimpo']:.3f}) | "
        f"{elapsed:.0f}s"
    )

## 4. Entrenar los 4 modelos sobre el split por bloques

In [ ]:
results = []
all_epochs_dfs = []
log_lines = [
    "TRAINING MODELS (SPLIT POR BLOQUES, v2) — RESULTADOS",
    f"Fecha: {datetime.now().strftime('%Y-%m-%d %H:%M')}",
    f"Device: {device} | GPU: {gpu_name}",
    f"PyTorch: {torch.__version__}",
    f"Corrida: {RUN_NAME} | Manifiestos: {MANIFEST_DIR}",
    f"Batch={BATCH_SIZE} | Max epochs={MAX_EPOCHS} | Patience={PATIENCE}",
    f"LR CNN={LR_CNN} | LR Transformer={LR_TRANSFORMER}",
    f"Train chips: {len(train_loader.dataset):,} | Val: {len(val_loader.dataset):,}",
    "",
]

for i, model_cfg in enumerate(MODELS, start=1):
    name = model_cfg["name"]
    label = model_cfg["label"]
    lr = model_cfg["lr"]

    print("\n" + "=" * 60)
    print(f"[{i}/{len(MODELS)}] Entrenando: {label} ({name}) | LR={lr}")
    print("=" * 60)

    log_lines += ["=" * 60, f"MODELO: {label} ({name}) | LR={lr}", "=" * 60, ""]

    summary = train_model(
        model_name=name,
        train_loader=train_loader,
        val_loader=val_loader,
        save_dir=MODELS_DIR,
        device=device,
        max_epochs=MAX_EPOCHS,
        patience=PATIENCE,
        lr=lr,
    )
    summary["model_label"] = label

    # CSV por epoch de este modelo
    epoch_df = history_to_dataframe(summary)
    epoch_csv = REPORT_DIR / f"epochs_{name}_{RUN_NAME}.csv"
    epoch_df.to_csv(epoch_csv, index=False)
    all_epochs_dfs.append(epoch_df)

    # Líneas de texto por epoch
    for tr, va in zip(summary["history"]["train"], summary["history"]["val"]):
        log_lines.append(format_epoch_line(
            tr["epoch"], MAX_EPOCHS, tr, va, tr.get("time_s", 0)
        ))

    val_history = summary["history"]["val"]
    best_val = next(r for r in val_history if r["epoch"] == summary["best_epoch"])

    log_lines += [
        "",
        f"→ Mejor epoch: {summary['best_epoch']} | val macro F1: {summary['best_val_macro_f1']:.4f}",
        f"→ val accuracy: {best_val['accuracy']:.4f} | val recall com_garimpo: {best_val['recall_com_garimpo']:.4f}",
        f"→ Checkpoint: {summary['checkpoint']}",
        f"→ CSV epochs: {epoch_csv}",
        "",
    ]

    plot_training_curves(summary, FIGURES_DIR / f"{name}_{RUN_NAME}_training_curves.png")

    results.append({
        "model": label,
        "timm_name": name,
        "variant": RUN_NAME,
        "lr": lr,
        "best_epoch": summary["best_epoch"],
        "epochs_run": len(summary["history"]["train"]),
        "val_macro_f1": round(summary["best_val_macro_f1"], 4),
        "val_accuracy": round(best_val["accuracy"], 4),
        "val_recall_com_garimpo": round(best_val["recall_com_garimpo"], 4),
        "val_f1_com_garimpo": round(best_val["f1_com_garimpo"], 4),
        "val_f1_sem_garimpo": round(best_val["f1_sem_garimpo"], 4),
        "checkpoint": summary["checkpoint"],
    })

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"\nEntrenamiento ({RUN_NAME}) completado.")

## 5. Guardar resumen (CSV + TXT)

In [ ]:
results_df = pd.DataFrame(results).sort_values("val_macro_f1", ascending=False)
display(results_df)

# CSV resumen comparativo
summary_csv = REPORT_DIR / f"training_results_summary_{RUN_NAME}.csv"
results_df.to_csv(summary_csv, index=False)

# CSV todas las epochs de los 4 modelos
all_epochs_csv = REPORT_DIR / f"training_epochs_all_models_{RUN_NAME}.csv"
pd.concat(all_epochs_dfs, ignore_index=True).to_csv(all_epochs_csv, index=False)

# TXT legible
best = results_df.iloc[0]
log_lines += [
    "=" * 60,
    f"RESUMEN COMPARATIVO (validation, {RUN_NAME})",
    "=" * 60,
    "",
    results_df.to_string(index=False),
    "",
    f"MEJOR MODELO ({RUN_NAME}): {best['model']} | val macro F1 = {best['val_macro_f1']}",
    f"Checkpoint: {best['checkpoint']}",
]

summary_txt = REPORT_DIR / f"training_results_{RUN_NAME}.txt"
summary_txt.write_text("\n".join(log_lines), encoding="utf-8")

print("\nArchivos guardados:")
print(f"  {summary_csv}")
print(f"  {all_epochs_csv}")
print(f"  {summary_txt}")
print(f"\nMejor modelo ({RUN_NAME}): {best['model']} (val F1={best['val_macro_f1']})")

## 6. Comparación: v1 tuned (split por ráster) vs. v2 (split por bloques)

Los dos conjuntos de validación son distintos, así que la diferencia mide sobre todo **cuánto
cambia la dificultad del problema** al repartir el territorio de otra forma. La comparación
concluyente entre modelos es la de test, en el notebook de evaluación.

In [ ]:
baseline_csv = DATA_DIR / "08_reporting" / "v2_bloques" / "training_results_summary_v2_bloques.csv"

if baseline_csv.exists():
    base_df = pd.read_csv(baseline_csv)[["timm_name", "val_macro_f1", "val_accuracy"]].rename(
        columns={
            "val_macro_f1": "val_macro_f1_v2_bloques",
            "val_accuracy": "val_accuracy_v2_bloques",
        }
    )
    tuned_df = results_df[["timm_name", "val_macro_f1", "val_accuracy"]].rename(
        columns={
            "val_macro_f1": "val_macro_f1_v2_bloques_tuned",
            "val_accuracy": "val_accuracy_v2_bloques_tuned",
        }
    )
    comparacion = base_df.merge(tuned_df, on="timm_name", how="outer")
    comparacion["f1_delta_tuned_menos_baseline"] = (
        comparacion["val_macro_f1_v2_bloques_tuned"] - comparacion["val_macro_f1_v2_bloques"]
    ).round(4)
    display(comparacion.sort_values("f1_delta_tuned_menos_baseline", ascending=False))

    compare_csv = REPORT_DIR / f"v2_bloques_vs_{RUN_NAME}_comparison.csv"
    comparacion.to_csv(compare_csv, index=False)
    print(f"\nGuardado: {compare_csv}")
else:
    print(f"No se encontró {baseline_csv}. Ejecuta antes 07_train_bloques.ipynb.")


## 7. Próximo paso

Evaluar estos checkpoints con `08c_eval_bloques_tuned.ipynb` sobre el mismo `manifest_test.csv`
de `v2_bloques`.
